# 🌾 KrishiRakshak — Model Training on Google Colab (Free T4 GPU)

This notebook trains the **MobileNetV2 (38 Classes)** PlantVillage disease classifier with PyTorch on Colab's GPU and downloads `best_model.pt` directly to your computer.

### ⚡ Step 0: Ensure GPU is Enabled
Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.

In [ ]:
# 1. Verify GPU availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Clone the KrishiRakshak repository
!git clone https://github.com/AkshithCodez/KrishiRakshak.git
%cd KrishiRakshak

In [ ]:
# 3. Download the 38-class PlantVillage Dataset
import os
os.makedirs("ml/data", exist_ok=True)

!pip install -q kagglehub
import kagglehub

print("Downloading PlantVillage dataset from Kaggle...")
path = kagglehub.dataset_download("emmarex/plantdisease")
print(f"Dataset downloaded to: {path}")

# Locate the PlantVillage folder
dataset_folder = os.path.join(path, "PlantVillage") if os.path.exists(os.path.join(path, "PlantVillage")) else path
print("Found dataset directory:", dataset_folder)
print("Classes found:", len(os.listdir(dataset_folder)))

In [ ]:
# 4. Run Training with MobileNetV2
# This takes ~15-20 minutes on Colab T4 GPU
!python ml/src/train.py \
    --data-dir "$dataset_folder" \
    --epochs 15 \
    --batch-size 32 \
    --head-lr 0.001 \
    --output-dir ml/models

In [ ]:
# 5. Export trained model to ONNX for fast inference
!python ml/src/export_onnx.py \
    --model-path ml/models/best_model.pt \
    --output ml/models/model.onnx

In [ ]:
# 6. Download best_model.pt and model.onnx directly to your local PC
from google.colab import files

print("Downloading best_model.pt...")
files.download('ml/models/best_model.pt')

if os.path.exists('ml/models/model.onnx'):
    print("Downloading model.onnx...")
    files.download('ml/models/model.onnx')

if os.path.exists('ml/models/class_info.json'):
    files.download('ml/models/class_info.json')